# North DFW Housing Affordability Crisis
## Notebook 01: Data Ingestion

**Author:** Alejandro Molina  
**GitHub:** https://github.com/chooseyourtacoday  
**Date:** 2026

---

### Business Context
A coalition of concerned residents in North DFW — Frisco, McKinney, Allen, Prosper, and Celina — commissioned this independent data analysis to understand why housing costs are spiraling out of control and what, if anything, can be done about it.

### Notebook Goal
Load, inspect, filter, and stage all raw datasets for downstream cleaning and analysis.

### Data Sources
| Dataset | Source | File |
|---------|--------|------|
| Home Values (ZHVI) | [Zillow Research](https://www.zillow.com/research/data/) | `Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv` |
| Mortgage Rates | [FRED](https://fred.stlouisfed.org/series/MORTGAGE30US) | `MORTGAGE30US.csv` |
| Median Household Income | [Texas DSHS](https://www.dshs.texas.gov/) | `indicator_data_download_20260427.csv` |
| Population Estimates | [Texas Demographic Center](https://demographics.texas.gov/) | `2024_txpopest_place.csv` |
| Building Permits | [Census Bureau](https://www.census.gov/construction/bps/) | `Building_Permits.csv` |
| Wages by County | [BLS QCEW](https://www.bls.gov/cew/downloadable-data-files.htm) | `20XX.annual 48085/48121*.csv` |

> **Note:** Raw data files are not included in this repository due to file size.  
> Download each file from the sources above and place them in the `data/raw/` directory.

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import glob

print('Libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')

Libraries loaded successfully.
Pandas version: 2.3.3


In [2]:
import os
print(os.listdir(os.getcwd()))

['.ipynb_checkpoints', '01_Data_Ingestion.ipynb', 'data', 'staging']


In [3]:
import os
import shutil

# Create data/raw directory
os.makedirs(os.path.join('data', 'raw'), exist_ok=True)

# List of files to move
files_to_move = [
    'Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv',
    'MORTGAGE30US.csv',
    'indicator_data_download_20260427.csv',
    '2024_txpopest_place.csv',
    'Building Permits.csv',
]

# Move main files
for f in files_to_move:
    if os.path.exists(f):
        shutil.move(f, os.path.join('data', 'raw', f))
        print(f'Moved: {f}')

# Move all BLS files
for f in os.listdir('.'):
    if f.endswith('.csv') and 'annual' in f:
        shutil.move(f, os.path.join('data', 'raw', f))
        print(f'Moved: {f}')

print('\nDone. Verifying data/raw contents:')
print(os.listdir(os.path.join('data', 'raw')))


Done. Verifying data/raw contents:
['2015.annual 48085 Collin County, Texas.csv', '2015.annual 48121 Denton County, Texas.csv', '2016.annual 48085 Collin County, Texas.csv', '2016.annual 48121 Denton County, Texas.csv', '2017.annual 48085 Collin County, Texas.csv', '2017.annual 48121 Denton County, Texas.csv', '2018.annual 48085 Collin County, Texas.csv', '2018.annual 48121 Denton County, Texas.csv', '2019.annual 48085 Collin County, Texas.csv', '2019.annual 48121 Denton County, Texas.csv', '2020.annual 48085 Collin County, Texas.csv', '2020.annual 48121 Denton County, Texas.csv', '2021.annual 48085 Collin County, Texas.csv', '2021.annual 48121 Denton County, Texas.csv', '2022.annual 48085 Collin County, Texas.csv', '2022.annual 48121 Denton County, Texas.csv', '2023.annual 48085 Collin County, Texas.csv', '2023.annual 48121 Denton County, Texas.csv', '2024.annual 48085 Collin County, Texas.csv', '2024.annual 48121 Denton County, Texas.csv', '2024_txpopest_place.csv', 'Building Permit

In [4]:
# Directory paths
RAW_DIR     = os.path.join('data', 'raw')
STAGING_DIR = 'staging'

os.makedirs(STAGING_DIR, exist_ok=True)

# File paths — matched to actual filenames
ZILLOW_FILE     = os.path.join(RAW_DIR, 'Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')
MORTGAGE_FILE   = os.path.join(RAW_DIR, 'MORTGAGE30US.csv')
INCOME_FILE     = os.path.join(RAW_DIR, 'indicator_data_download_20260427.csv')
POPULATION_FILE = os.path.join(RAW_DIR, '2024_txpopest_place.csv')
PERMITS_FILE    = os.path.join(RAW_DIR, 'Building Permits.csv')
BLS_PATTERN     = os.path.join(RAW_DIR, '20*.annual 48*.csv')

print(f'Raw data directory : {RAW_DIR}')
print(f'Staging directory  : {STAGING_DIR}')

Raw data directory : data\raw
Staging directory  : staging


## Step 2: Configure File Paths

All raw data files should be placed in the `data/raw/` directory relative to this notebook.  
Staged outputs will be saved to the `staging/` directory.

In [5]:
# Directory paths
RAW_DIR     = os.path.join('data', 'raw')
STAGING_DIR = 'staging'

os.makedirs(STAGING_DIR, exist_ok=True)

# File paths
ZILLOW_FILE     = os.path.join(RAW_DIR, 'Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')
MORTGAGE_FILE   = os.path.join(RAW_DIR, 'MORTGAGE30US.csv')
INCOME_FILE     = os.path.join(RAW_DIR, 'indicator_data_download.csv')
POPULATION_FILE = os.path.join(RAW_DIR, '2024_txpopest_place.csv')
PERMITS_FILE    = os.path.join(RAW_DIR, 'Building_Permits.csv')
BLS_PATTERN     = os.path.join(RAW_DIR, '20*.annual 48*.csv.csv')

print(f'Raw data directory : {RAW_DIR}')
print(f'Staging directory  : {STAGING_DIR}')

Raw data directory : data\raw
Staging directory  : staging


## Step 3: Define Target Geography

In [6]:
north_dfw_zips = [
    75033, 75034, 75035,
    75069, 75070, 75071,
    75002, 75013,
    75078,
    75009,
    75023, 75024, 75025
]

zip_city_map = {
    75033: 'Frisco',   75034: 'Frisco',   75035: 'Frisco',
    75069: 'McKinney', 75070: 'McKinney', 75071: 'McKinney',
    75002: 'Allen',    75013: 'Allen',
    75078: 'Prosper',
    75009: 'Celina',
    75023: 'Plano',    75024: 'Plano',    75025: 'Plano'
}

our_cities   = ['Frisco', 'McKinney', 'Allen', 'Prosper', 'Celina', 'Plano']
our_counties = ['Collin County, Texas', 'Denton County, Texas']

print(f'Target zip codes : {len(north_dfw_zips)}')
print(f'Target cities    : {our_cities}')

Target zip codes : 13
Target cities    : ['Frisco', 'McKinney', 'Allen', 'Prosper', 'Celina', 'Plano']


In [7]:
zillow_raw = pd.read_csv(ZILLOW_FILE)
print(f'Zillow raw shape: {zillow_raw.shape}')

zillow_dfw = zillow_raw[zillow_raw['RegionName'].isin(north_dfw_zips)].copy()
print(f'Filtered to North DFW: {zillow_dfw.shape[0]} zip codes')
print(zillow_dfw[['RegionName', 'City', 'CountyName']].to_string(index=False))

Zillow raw shape: (26283, 324)
Filtered to North DFW: 13 zip codes
 RegionName     City    CountyName
      75035   Frisco Collin County
      75002    Allen Collin County
      75071 McKinney Collin County
      75070 McKinney Collin County
      75025    Plano Collin County
      75033   Frisco Denton County
      75023    Plano Collin County
      75034   Frisco Collin County
      75013    Allen Collin County
      75024    Plano Collin County
      75078  Prosper Collin County
      75069 McKinney Collin County
      75009   Celina Collin County


## Step 4: Load Zillow Home Value Index (ZHVI)

In [8]:
zillow_raw = pd.read_csv(ZILLOW_FILE)
print(f'Zillow raw shape: {zillow_raw.shape}')

zillow_dfw = zillow_raw[zillow_raw['RegionName'].isin(north_dfw_zips)].copy()
print(f'Filtered to North DFW: {zillow_dfw.shape[0]} zip codes')
print(zillow_dfw[['RegionName', 'City', 'CountyName']].to_string(index=False))

Zillow raw shape: (26283, 324)
Filtered to North DFW: 13 zip codes
 RegionName     City    CountyName
      75035   Frisco Collin County
      75002    Allen Collin County
      75071 McKinney Collin County
      75070 McKinney Collin County
      75025    Plano Collin County
      75033   Frisco Denton County
      75023    Plano Collin County
      75034   Frisco Collin County
      75013    Allen Collin County
      75024    Plano Collin County
      75078  Prosper Collin County
      75069 McKinney Collin County
      75009   Celina Collin County


In [9]:
meta_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
             'StateName', 'State', 'City', 'Metro', 'CountyName']
date_cols = [col for col in zillow_dfw.columns if col not in meta_cols]

print(f'Date columns : {len(date_cols)}')
print(f'Date range   : {date_cols[0]} to {date_cols[-1]}')

Date columns : 315
Date range   : 1/31/2000 to 3/31/2026


In [10]:
# Melt wide to long format
zillow_long = zillow_dfw.melt(
    id_vars=meta_cols,
    value_vars=date_cols,
    var_name='date',
    value_name='home_value'
)

zillow_long['date'] = pd.to_datetime(zillow_long['date'])
zillow_long['city'] = zillow_long['RegionName'].map(zip_city_map)
zillow_long = zillow_long.sort_values(['RegionName', 'date']).reset_index(drop=True)

# Filter to 2015-present
zillow_filtered = zillow_long[zillow_long['date'] >= '2015-01-01'].copy()

print(f'Long format rows    : {zillow_long.shape[0]}')
print(f'Filtered 2015+      : {zillow_filtered.shape[0]} rows')
print(f'Date range          : {zillow_filtered["date"].min().date()} to {zillow_filtered["date"].max().date()}')
print(f'Nulls in home_value : {zillow_filtered["home_value"].isnull().sum()}')

Long format rows    : 4095
Filtered 2015+      : 1755 rows
Date range          : 2015-01-31 to 2026-03-31
Nulls in home_value : 0


In [11]:
print('=== HOME VALUE SUMMARY BY CITY (2015-Present) ===')
print(zillow_filtered.groupby('city')['home_value'].describe().round(0))

=== HOME VALUE SUMMARY BY CITY (2015-Present) ===
          count      mean       std       min       25%       50%       75%  \
city                                                                          
Allen     270.0  450954.0  132701.0  236698.0  320662.0  434715.0  495676.0   
Celina    135.0  443846.0  117452.0  278315.0  355995.0  367528.0  578555.0   
Frisco    405.0  529140.0  129223.0  334990.0  420467.0  449227.0  680298.0   
McKinney  405.0  383880.0   94496.0  214971.0  307280.0  335359.0  462501.0   
Plano     405.0  436213.0  120733.0  216295.0  356232.0  427819.0  535242.0   
Prosper   135.0  596653.0  160728.0  400507.0  465450.0  481372.0  783441.0   

               max  
city                
Allen     706461.0  
Celina    627632.0  
Frisco    728579.0  
McKinney  557542.0  
Plano     671855.0  
Prosper   846895.0  


In [12]:
zillow_filtered.to_csv(os.path.join(STAGING_DIR, 'zillow_dfw_2015_present.csv'), index=False)
print('Saved: staging/zillow_dfw_2015_present.csv')

Saved: staging/zillow_dfw_2015_present.csv


## Step 5: Load FRED Mortgage Rates

In [13]:
mortgage_raw = pd.read_csv(MORTGAGE_FILE)
mortgage_raw.columns = ['date', 'mortgage_rate']
mortgage_raw['date'] = pd.to_datetime(mortgage_raw['date'])

mortgage_filtered = mortgage_raw[mortgage_raw['date'] >= '2015-01-01'].copy()
mortgage_monthly  = mortgage_filtered.set_index('date').resample('ME').mean().reset_index()
mortgage_monthly.columns = ['date', 'mortgage_rate']

print(f'Monthly rows : {mortgage_monthly.shape[0]}')
print(f'Date range   : {mortgage_monthly["date"].min().date()} to {mortgage_monthly["date"].max().date()}')
print(f'Nulls        : {mortgage_monthly.isnull().sum().sum()}')

mortgage_monthly.to_csv(os.path.join(STAGING_DIR, 'mortgage_rates_monthly.csv'), index=False)
print('Saved: staging/mortgage_rates_monthly.csv')

Monthly rows : 136
Date range   : 2015-01-31 to 2026-04-30
Nulls        : 0
Saved: staging/mortgage_rates_monthly.csv


## Step 6: Load Median Household Income

In [14]:
INCOME_FILE = os.path.join(RAW_DIR, 'indicator_data_download_20260427.csv')
print(f'Income file: {INCOME_FILE}')

Income file: data\raw\indicator_data_download_20260427.csv


In [15]:
income_raw = pd.read_csv(INCOME_FILE)

income_clean = income_raw[
    (income_raw['Location'].isin(our_cities)) &
    (income_raw['Breakout Category'].isna())
][['Location', 'Indicator Rate Value', 'Period of Measure']].copy()

income_clean.columns = ['city', 'median_income', 'period']
income_clean = income_clean.drop_duplicates().reset_index(drop=True)

print(f'Income rows : {income_clean.shape[0]}')
print(f'Cities      : {income_clean["city"].unique().tolist()}')

income_clean.to_csv(os.path.join(STAGING_DIR, 'income_by_city.csv'), index=False)
print('Saved: staging/income_by_city.csv')

Income rows : 46
Cities      : ['Allen', 'Celina', 'Frisco', 'McKinney', 'Plano', 'Prosper']
Saved: staging/income_by_city.csv


## Step 7: Load Population Estimates

POPULATION_FILE = os.path.join(RAW_DIR, '2024_txpopest_place.csv.xlsx')
print(f'Population file: {POPULATION_FILE}')

import os
print(os.listdir(os.path.join('data', 'raw')))

In [16]:
population_raw = pd.read_csv(POPULATION_FILE)

population_clean = population_raw[
    population_raw['Place'].isin(our_cities)
][['Place', 'census_2020_count', 'july1_2021_pop_est',
   'july1_2022_pop_est', 'july1_2023_pop_est',
   'july1_2024_pop_est', 'jan1_2025_pop_est']].copy()

population_clean.columns = ['city', 'pop_2020', 'pop_2021',
                             'pop_2022', 'pop_2023', 'pop_2024', 'pop_2025']

print(f'Population rows : {population_clean.shape[0]}')
print(population_clean.to_string(index=False))

population_clean.to_csv(os.path.join(STAGING_DIR, 'population_by_city.csv'), index=False)
print('\nSaved: staging/population_by_city.csv')

Population rows : 6
    city  pop_2020  pop_2021  pop_2022  pop_2023  pop_2024  pop_2025
   Allen  104627.0  107421.0    109169    110983    112155    112748
  Celina   16739.0   23712.0     29971     38988     44929     47908
  Frisco  200509.0  211966.0    216858    220972    234038    240561
McKinney  195308.0  202267.0    212944    218559    226004    229734
   Plano  285494.0  288735.0    292452    294292    293286    292788
 Prosper   30174.0   34070.0     38569     41385     43519     44586

Saved: staging/population_by_city.csv


## Step 8: Load Building Permits

In [17]:
PERMITS_FILE = os.path.join(RAW_DIR, 'Building Permits.csv')
permits_raw = pd.read_csv(PERMITS_FILE)
permits_raw.columns = ['geo', 'frequency', 'date', 'units_1unit',
                        'avg_value_1unit', 'buildings_5plus',
                        'units_5plus', 'avg_bldg_value_5plus',
                        'avg_unit_value_5plus']
permits_raw['date'] = pd.to_datetime(permits_raw['date'])
permits_clean = permits_raw[permits_raw['date'] >= '2015-01-01'].copy()
print(f'Permits rows : {permits_clean.shape[0]}')
permits_clean.to_csv(os.path.join(STAGING_DIR, 'permits_dfw_monthly.csv'), index=False)
print('Saved: staging/permits_dfw_monthly.csv')

Permits rows : 133
Saved: staging/permits_dfw_monthly.csv


In [18]:
BLS_PATTERN = os.path.join(RAW_DIR, '20*.annual 48*.csv')
bls_files = glob.glob(BLS_PATTERN)
print(f'BLS files found: {len(bls_files)}')

BLS files found: 20


In [19]:
PERMITS_FILE = os.path.join(RAW_DIR, 'Building Permits.csv')

In [20]:
permits_raw = pd.read_csv(PERMITS_FILE)
permits_raw.columns = ['geo', 'frequency', 'date', 'units_1unit',
                        'avg_value_1unit', 'buildings_5plus',
                        'units_5plus', 'avg_bldg_value_5plus',
                        'avg_unit_value_5plus']

permits_raw['date'] = pd.to_datetime(permits_raw['date'])
permits_clean = permits_raw[permits_raw['date'] >= '2015-01-01'].copy()

print(f'Permits rows : {permits_clean.shape[0]}')
print(f'Date range   : {permits_clean["date"].min().date()} to {permits_clean["date"].max().date()}')

permits_clean.to_csv(os.path.join(STAGING_DIR, 'permits_dfw_monthly.csv'), index=False)
print('Saved: staging/permits_dfw_monthly.csv')

Permits rows : 133
Date range   : 2015-01-01 to 2026-01-01
Saved: staging/permits_dfw_monthly.csv


## Step 9: Load BLS Wage Data
Annual average wages for Collin County (FIPS 48085) and Denton County (FIPS 48121), 2015-2024.

In [21]:
bls_files = glob.glob(BLS_PATTERN)
print(f'BLS files found: {len(bls_files)}')

dfs = []
for f in bls_files:
    df = pd.read_csv(f)
    df['industry_code'] = df['industry_code'].astype(str).str.strip()
    filtered = df[
        (df['own_code'] == 0) &
        (df['industry_code'] == '10') &
        (df['agglvl_code'] == 70)
    ][['area_fips', 'year', 'area_title',
       'annual_avg_wkly_wage', 'avg_annual_pay']].copy()
    dfs.append(filtered)

bls_annual = pd.concat(dfs, ignore_index=True)
bls_annual.columns = ['area_fips', 'year', 'county',
                       'avg_weekly_wage', 'avg_annual_pay']
bls_annual = bls_annual.sort_values(['county', 'year']).reset_index(drop=True)

print(f'BLS rows : {bls_annual.shape[0]}')
print(bls_annual.to_string(index=False))

bls_annual.to_csv(os.path.join(STAGING_DIR, 'bls_wages_annual.csv'), index=False)
print('\nSaved: staging/bls_wages_annual.csv')

BLS files found: 20
BLS rows : 20
 area_fips  year               county  avg_weekly_wage  avg_annual_pay
     48085  2015 Collin County, Texas             1185           61631
     48085  2016 Collin County, Texas             1207           62788
     48085  2017 Collin County, Texas             1233           64123
     48085  2018 Collin County, Texas             1286           66886
     48085  2019 Collin County, Texas             1315           68366
     48085  2020 Collin County, Texas             1415           73602
     48085  2021 Collin County, Texas             1495           77726
     48085  2022 Collin County, Texas             1560           81114
     48085  2023 Collin County, Texas             1602           83282
     48085  2024 Collin County, Texas             1694           88091
     48121  2015 Denton County, Texas              907           47176
     48121  2016 Denton County, Texas              935           48644
     48121  2017 Denton County, Texas      

## Step 10: Ingestion Summary

In [22]:
staging_files = {
    'zillow_dfw_2015_present.csv' : 'Home values by zip code (monthly, 2015-2026)',
    'mortgage_rates_monthly.csv'  : 'FRED 30yr mortgage rates (monthly, 2015-2026)',
    'income_by_city.csv'          : 'Median household income by city (ACS 5-yr)',
    'population_by_city.csv'      : 'Population estimates by city (TDC, 2020-2025)',
    'permits_dfw_monthly.csv'     : 'Building permits Dallas metro (monthly, 2015-2026)',
    'bls_wages_annual.csv'        : 'Avg wages Collin & Denton County (annual, 2015-2024)'
}

print('=' * 60)
print('NOTEBOOK 01 COMPLETE — INGESTION SUMMARY')
print('=' * 60)

for fname, desc in staging_files.items():
    df = pd.read_csv(os.path.join(STAGING_DIR, fname))
    print(f'\n{fname}')
    print(f'  Description : {desc}')
    print(f'  Rows        : {df.shape[0]}')
    print(f'  Columns     : {df.shape[1]}')

print('\n' + '=' * 60)
print('Next: Notebook 02 — Data Cleaning & Wrangling')
print('=' * 60)

NOTEBOOK 01 COMPLETE — INGESTION SUMMARY

zillow_dfw_2015_present.csv
  Description : Home values by zip code (monthly, 2015-2026)
  Rows        : 1755
  Columns     : 12

mortgage_rates_monthly.csv
  Description : FRED 30yr mortgage rates (monthly, 2015-2026)
  Rows        : 136
  Columns     : 2

income_by_city.csv
  Description : Median household income by city (ACS 5-yr)
  Rows        : 46
  Columns     : 3

population_by_city.csv
  Description : Population estimates by city (TDC, 2020-2025)
  Rows        : 6
  Columns     : 7

permits_dfw_monthly.csv
  Description : Building permits Dallas metro (monthly, 2015-2026)
  Rows        : 133
  Columns     : 9

bls_wages_annual.csv
  Description : Avg wages Collin & Denton County (annual, 2015-2024)
  Rows        : 20
  Columns     : 5

Next: Notebook 02 — Data Cleaning & Wrangling
